# Actualización del histórico



In [ ]:
# Actualizacion historico de leads

import os

from_drive = True  # same flag you use everywhere

if os.environ.get("HERMES_BOOTSTRAPPED") != "1":
    # ---------- GIT ON COLAB ONLY ----------
    try:
        from google.colab import userdata

        git_token = userdata.get('gitToken')
        git_user = userdata.get('gitUser')
        git_url = f'https://{git_token}@github.com/rene-aum/Hermes.git'
        branch_to_pull = 'dev'

        os.chdir('/content')

        if not os.path.isdir('Hermes'):
            !git clone {git_url}

        %cd Hermes
        !git fetch origin {branch_to_pull}
        !git checkout {branch_to_pull}
        !git pull origin {branch_to_pull}

        !pip install -r utils/src/requirements.txt
        %cd Asignacion

    except Exception as e:
        print(e)
        print('Running in other environment not colab probably!')

    # ---------- DRIVE + SHEETS ----------
    if from_drive:
        from pydrive2.auth import GoogleAuth
        from pydrive2.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        import gdown

        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)

        creds, _ = default()
        gc = gspread.authorize(creds)

    os.environ["HERMES_BOOTSTRAPPED"] = "1"
else:
    print("Bootstrap already done, assuming orchestrator ran it.")


In [ ]:

import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
import sys
import pytz
sys.path.append('..')
sys.path.append('../..')
from utils.utils import (get_dates_dataframe,
                       add_year_week,
                       custom_read,
                       process_columns,
                       remove_accents)

from utils.drive_toolbox import(from_drive_to_local,
                             get_last_modification_date_drive,
                             create_sheets_in_drive_folder,
                             update_sheets_in_drive_folder,
                             read_from_google_sheets,
                             list_file_ids_for_drive_folder,
                             create_csv_file_in_drive_folder,
                             write_csv_to_drive,
                             read_csv_from_drive)
from utils.src.constants import (atlas_consumo_output_folder_id,
                           consumo_sheets_ids_dict,
                           folder_id_bauto_gabo,
                           id_reporte_ventas,
                           id_edas_referenciados,
                           id_torre_de_control
                           )


warnings.filterwarnings('ignore')

In [ ]:
mexico_city_tz = pytz.timezone('America/Mexico_City')
pd.set_option('display.max_columns', None)
today = datetime.now(tz=mexico_city_tz).strftime('%Y-%m-%d %H:%M')
today

In [ ]:
id_folder_historico_lead = '1zvW-Dxow9gz1Dnbpg_jO7my4wadDvJDW'
dicc_historico = list_file_ids_for_drive_folder(drive,id_folder_historico_lead)
historico = read_csv_from_drive(drive,'1EVBy847HLGatCkN8pNqbb44pe_ULEBCS')
cols_borrar = historico.filter(like = 'reactivacion').columns.values.tolist()
historico = historico.drop(columns = cols_borrar)

In [ ]:
# torre legacy
torre_1_raw = read_from_google_sheets(gc,id_torre_de_control,sheetname='Asignación compradores')
print(f'shape historico: {historico.shape[0]} \nLeads únicos en histórico: {historico['id lead'].nunique()}')


## update con torre legacy


In [ ]:

torre_1_open = (torre_1_raw
 .pipe(process_columns)
 [['id_lead','origen_automarket','cosecha','id_comprador','folio_bauto','nombre_comprador','mail_comprador', 'telefono_comprador','espacio_automarket','asesor_de_ventas', 'fecha_de_asignacion','estatus_de_lead']]
 .assign(fecha_de_asignacion = lambda x: pd.to_datetime(x['fecha_de_asignacion'],format='%d/%m/%Y',errors='coerce').dt.strftime('%Y-%m-%d'))
 [lambda x: ~x.estatus_de_lead.isin(['PRUEBA'])]
 )
torre_1_open = (torre_1_open.rename(columns={'asesor_de_ventas':'asesor_espacio',
                                             'folio_bauto':'folio_bauto_tc'})
                )
torre_1_open.columns = [x.lower().replace('_',' ') for x in torre_1_open.columns]
torre_1_open = torre_1_open.rename(columns={'folio_bauto':'folio bauto tc'})

# en la torre legacy ya no vale reasignar leads cerrados

'Leads abiertos torre legacy',torre_1_open[lambda x: ~x['estatus de lead'].isin(['CERRADO','PRUEBA','COMPRA EXITOSA','COMPRA EXITOSA '])].shape[0]

In [ ]:
print(f'shape torre vieja: {torre_1_raw.shape[0]}. leads únicos torre vieja: {torre_1_raw['ID Lead'].nunique()}')
assert torre_1_raw.shape[0] == torre_1_raw['ID Lead'].nunique(), 'Leads duplicados en torre vieja'

In [ ]:

# actualizar los leads abiertos
update_tcv1_df = (historico[lambda x: ~x['estatus de lead'].isin(['CERRADO','PRUEBA','COMPRA EXITOSA','COMPRA EXITOSA '])]
 .merge(torre_1_open[['id lead','estatus de lead','asesor espacio','espacio automarket']],on='id lead',how='left',suffixes=('',' tcv1'))
 [lambda x: (x['estatus de lead tcv1'].notna())&
            (
                 (x['estatus de lead'].str.strip()!=x['estatus de lead tcv1'].str.strip())
             |
              (x['asesor espacio'].str.strip()!=x['asesor espacio tcv1'].str.strip())
            )
 ]
 )
print(f'leads abiertos a actualizar: {update_tcv1_df.shape[0]}')
# paso actualizacion
update_tcv1_df['estatus de lead']  = update_tcv1_df['estatus de lead tcv1']
update_tcv1_df['asesor espacio'] = update_tcv1_df['asesor espacio tcv1']
update_tcv1_df['espacio automarket'] = update_tcv1_df['espacio automarket tcv1']
update_tcv1_df['fecha_de_proceso'] = datetime.now(mexico_city_tz).strftime('%Y-%m-%d %H:%M:%S')
update_tcv1_df = update_tcv1_df.drop(columns=['estatus de lead tcv1','asesor espacio tcv1','espacio automarket tcv1'])


In [ ]:

# actualizar los leads cerrados
update_tcv1_cerr = (historico[lambda x: x['estatus de lead'].isin(['CERRADO','COMPRA EXITOSA','COMPRA EXITOSA '])]
 .merge(torre_1_open[['id lead','estatus de lead','asesor espacio','espacio automarket']],on='id lead',how='left',suffixes=('',' tcv1'))
 [lambda x: (x['estatus de lead tcv1'].notna())&
            ((x['estatus de lead'].str.strip()!=x['estatus de lead tcv1'].str.strip())
                | (x['espacio automarket'].str.strip()!=x['espacio automarket tcv1'].str.strip().str.title())
             | (x['asesor espacio'].str.strip()!=x['asesor espacio tcv1'].str.strip().str.title())
            )
 ]
 )
print(f'leads cerrados tcv1 a actualizar: {update_tcv1_cerr.shape[0]}')
# paso actualizacion
update_tcv1_cerr['estatus de lead']  = update_tcv1_cerr['estatus de lead tcv1']
update_tcv1_cerr['asesor espacio'] = update_tcv1_cerr['asesor espacio tcv1']
update_tcv1_cerr['espacio automarket'] = update_tcv1_cerr['espacio automarket tcv1']
update_tcv1_cerr['fecha_de_proceso'] = datetime.now(mexico_city_tz).strftime('%Y-%m-%d %H:%M:%S')
update_tcv1_cerr = update_tcv1_cerr.drop(columns=['estatus de lead tcv1','asesor espacio tcv1','espacio automarket tcv1'])


In [ ]:
# extracto del historico que no se actualizara
df_no_update = (historico
 .merge(update_tcv1_df[['id lead']], on=['id lead'], how="left", indicator=True)
 [lambda x:x['_merge']=='left_only']
 .drop(columns=['_merge'])
  .merge(update_tcv1_cerr[['id lead']], on=['id lead'], how="left", indicator=True)
 [lambda x:x['_merge']=='left_only']
 .drop(columns=['_merge'])
 )

final_hist_v1 = (pd.concat([df_no_update,update_tcv1_df,update_tcv1_cerr])
 .reset_index(drop=True)

)

print(f'historico shape:{historico.shape[0]}, final hist v1 shape:{final_hist_v1.shape[0]}')
assert final_hist_v1.shape[0] == historico.shape[0]


## update con nueva torre de control


In [ ]:

id_torre_v2 = '1k8rguLeF1O33XCaVDxPiQ1C4SbxLDSIeqNcriYtsF-k'
asig_torre_v2 = read_from_google_sheets(gc,id_torre_v2,sheetname='asignacion')
asig_torre_v2['fecha de asignacion'] =  pd.to_datetime(asig_torre_v2['fecha de asignacion'],format='%d/%m/%Y',errors='coerce').dt.strftime('%Y-%m-%d')
subset_columns_v2 = ['id lead', 'origen automarket', 'cosecha', 'id comprador',
       'folio bauto tc', 'nombre comprador', 'mail comprador',
       'telefono comprador', 'asesor credito', 'espacio automarket',
       'asesor espacio', 'fecha de asignacion', 'estatus de lead',
        'fecha de reactivacion credito','fecha de reactivacion eam']



In [ ]:
print(f'shape torre v2: {asig_torre_v2.shape[0]}. leads únicos torre v2: {asig_torre_v2['id lead'].nunique()}')
assert asig_torre_v2.shape[0] == asig_torre_v2['id lead'].nunique(), 'Leads duplicados en torre v2'

In [ ]:

# append
nueva_torre_append = (asig_torre_v2[subset_columns_v2]
 .merge(final_hist_v1[['id lead']],on=['id lead'],how='left',indicator=True)
 [lambda x: x['_merge']=='left_only']
 .drop(columns=['_merge'])
 .assign(flag_torre_v2 = 1,
         fecha_de_proceso= lambda x: datetime.now(mexico_city_tz).strftime('%Y-%m-%d %H:%M:%S'))
 )
print(f'Filas a añadir de tc v2: {nueva_torre_append.shape[0]}')


In [ ]:

# update leads cerrados
update_df_torre_v2_cerr = (final_hist_v1
                           [lambda x: x['estatus de lead'].isin(['CERRADO','COMPRA EXITOSA','COMPRA EXITOSA '])]
 .merge(asig_torre_v2[['id lead','estatus de lead','asesor espacio','asesor credito','espacio automarket']],on='id lead',how='left',suffixes=('',' tcv2'),indicator=True)
 [lambda x: x['_merge']=='both']
.drop(columns=['_merge'])
 [lambda x: (x['estatus de lead tcv2'].notna())&
            (
                (x['estatus de lead'].str.strip()!=x['estatus de lead tcv2'].str.strip())
              |(x['espacio automarket'].str.strip()!=x['espacio automarket tcv2'].str.strip())
             | (x['asesor espacio'].str.strip()!=x['asesor espacio tcv2'].str.strip())
            )
 ]
.assign(flag_torre_v2 = 1,
        fecha_de_proceso = lambda x: datetime.now(mexico_city_tz).strftime('%Y-%m-%d %H:%M:%S'))
 )
print(f'Filas a actualizar de tc v2 leads cerrados: {update_df_torre_v2_cerr.shape[0]}')

update_df_torre_v2_cerr['flag salio de cerrado'] = (np.where((update_df_torre_v2_cerr['estatus de lead']=='CERRADO')&
                                                              (update_df_torre_v2_cerr['estatus de lead tcv2']!='CERRADO'),
                                                             1,
                                                             np.nan
                                                             )
                                                      )
update_df_torre_v2_cerr['espacio automarket'] = update_df_torre_v2_cerr['espacio automarket tcv2']
update_df_torre_v2_cerr['asesor espacio'] = update_df_torre_v2_cerr['asesor espacio tcv2']
update_df_torre_v2_cerr['asesor credito'] = update_df_torre_v2_cerr['asesor credito tcv2']
update_df_torre_v2_cerr['estatus de lead'] = update_df_torre_v2_cerr['estatus de lead tcv2']
update_df_torre_v2_cerr = update_df_torre_v2_cerr.drop(columns=['estatus de lead tcv2','asesor espacio tcv2','espacio automarket tcv2','asesor credito tcv2'])



In [ ]:

# update leads abiertos
update_df_torre_v2_open = (final_hist_v1
                           [lambda x: ~x['estatus de lead'].isin(['CERRADO','COMPRA EXITOSA','COMPRA EXITOSA '])]
 .merge(asig_torre_v2[['id lead','estatus de lead','asesor espacio','asesor credito','espacio automarket']],on='id lead',how='left',suffixes=('',' tcv2'),indicator=True)
 [lambda x: x['_merge']=='both']
.drop(columns=['_merge'])
 [lambda x: (x['estatus de lead tcv2'].notna())&
            (
    (x['estatus de lead'].str.strip()!=x['estatus de lead tcv2'].str.strip())
                 |(x['espacio automarket'].str.strip()!=x['espacio automarket tcv2'].str.strip())
             | (x['asesor espacio'].str.strip()!=x['asesor espacio tcv2'].str.strip())
            )
 ]
.assign(flag_torre_v2 = 1,
        fecha_de_proceso = lambda x: datetime.now(mexico_city_tz).strftime('%Y-%m-%d %H:%M:%S'))
 )
print(f'Filas a actualizar de tc v2 leads abiertos: {update_df_torre_v2_open.shape[0]}')
update_df_torre_v2_open['espacio automarket'] = update_df_torre_v2_open['espacio automarket tcv2']
update_df_torre_v2_open['asesor espacio'] = update_df_torre_v2_open['asesor espacio tcv2']
update_df_torre_v2_open['asesor credito'] = update_df_torre_v2_open['asesor credito tcv2']
update_df_torre_v2_open['estatus de lead'] = update_df_torre_v2_open['estatus de lead tcv2']
update_df_torre_v2_open = update_df_torre_v2_open.drop(columns=['estatus de lead tcv2','asesor espacio tcv2','espacio automarket tcv2','asesor credito tcv2'])

df_no_update_tc_v2 = (final_hist_v1
 .merge(update_df_torre_v2_open[['id lead']],on=['id lead'],how='left',indicator=True)
 [lambda x: x['_merge']=='left_only']
 .drop(columns=['_merge'])
  .merge(update_df_torre_v2_cerr[['id lead']],on=['id lead'],how='left',indicator=True)
 [lambda x: x['_merge']=='left_only']
 .drop(columns=['_merge'])
                      )


In [ ]:

historico.shape[0],final_hist_v1.shape[0],nueva_torre_append.shape[0]



## HISTORICO FINAL


In [ ]:


resultado_final = (pd.concat([
    df_no_update_tc_v2,
            nueva_torre_append,
            update_df_torre_v2_open,
            update_df_torre_v2_cerr
                              ])
.reset_index(drop=True)

)

resultado_final.shape[0],resultado_final['id lead'].nunique()


In [ ]:

print(resultado_final.shape[0],final_hist_v1.shape[0] + nueva_torre_append.shape[0] )
assert resultado_final.shape[0] == final_hist_v1.shape[0] + nueva_torre_append.shape[0]



## Write


In [ ]:
id_output_historico = dicc_historico['historico_tc_latest.csv']
create_csv_file_in_drive_folder(drive,id_folder_historico_lead,resultado_final,f'historico_tc_{today}.csv')

In [ ]:

write_csv_to_drive(drive,id_output_historico,resultado_final)

## free memory

In [ ]:
vars_to_del = ['id_folder_historico_lead','id_output_historico',
               'historico','dicc_historico','torre_1_raw','torre_1_open',
               'update_df_torre_v2_cerr','update_df_torre_v2_open',
               'update_tcv1_cerr','update_tcv1_df','final_hist_v1','asig_torre_v2','nueva_torre_append','resultado_final',
               'df_no_update','df_no_update_tc_v2']
for v in vars_to_del:
    try:
        del globals()[v]
    except Exception as e:
        print(f"could not delete var {v}: {e}")

In [ ]:
import gc as gcol
gcol.collect()